In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_finalproject")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlspchasiquiza01")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/sales_v2.csv"

In [0]:
df_products = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
schema_sales = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("order_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])

In [0]:
df_sales_final = spark.read\
.option('header', True)\
.schema(schema_sales)\
.csv(ruta)

In [0]:
sales_final_df = df_sales_final.withColumn("ingestion_date", current_timestamp())

In [0]:
sales_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.sales_bz")